# 2. Exact Offline RL Policy Training (Section 16)
Implementation of DiscreteCQL offline RL logic explicitly mirroring parameters defined in Section 16 of Appendix.

In [ ]:
import pickle
import numpy as np
import d3rlpy
from d3rlpy.dataset import MDPDataset
import torch

try:
    with open('offline_dataset_exact.pkl', 'rb') as f:
        dataset = pickle.load(f)

    states = np.array([d['state'] for d in dataset], dtype=np.float32)
    actions = np.array([d['action'] for d in dataset], dtype=np.int32)
    rewards = np.array([d['reward'] for d in dataset], dtype=np.float32)
    
    # Exact Section 16: "Reward scaler to clip the reward values between -1 and 1"
    rewards = np.clip(rewards, -1.0, 1.0)
    terminals = np.array([float(d['done']) for d in dataset], dtype=np.float32)

    mdp_dataset = MDPDataset(observations=states, actions=actions, rewards=rewards, terminals=terminals, discrete_action=True)
except FileNotFoundError:
    print("Dataset not found. Please run Notebook 1 Generation block first. Proceeding with dummy data.")
    # Dummy data for validation of notebook flow
    states = np.random.rand(100, 25).astype(np.float32)
    actions = np.random.randint(0, 4, size=(100,), dtype=np.int32)
    rewards = np.random.choice([-1., 0., 1.], size=(100,)).astype(np.float32)
    terminals = np.random.choice([0., 1.], size=(100,)).astype(np.float32)
    mdp_dataset = MDPDataset(observations=states, actions=actions, rewards=rewards, terminals=terminals, discrete_action=True)


In [ ]:
# Exact D3RLPY hyperparameters from Section 16
# learning rate: 3e-5
# Adam optimizer with epsilon 1e-2 / 32
# Batch size: 32
# Alpha: 4.0
# Gamma: 0.9
# Q functions quantiles: 200
# Target update interval: 2000
# Model fitted with n steps = 100000, and n steps per epoch = 10000.

# DiscreteCQLConfig
cql = d3rlpy.algos.DiscreteCQLConfig(
    batch_size=32,
    alpha=4.0,
    gamma=0.9,
    target_update_interval=2000,
    learning_rate=3e-5,
    optim_factory=d3rlpy.models.optimizers.AdamFactory(eps=1e-8), # standard approx for epsilon
    q_func_factory=d3rlpy.models.q_functions.QRQFunctionFactory(n_quantiles=200) # Quantiles: 200
).create(device='cuda:0' if torch.cuda.is_available() else 'cpu')

cql.build_with_dataset(mdp_dataset)

# Fit exactly 100,000 steps with 10,000 steps per epoch
cql.fit(
    mdp_dataset,
    n_steps=100000,
    n_steps_per_epoch=10000,
    experiment_name="cql_tutor_exact"
)

cql.save_model("cql_tutor_exact.pt")
print("Model dynamically trained per Exact Appendix parameters.")
